<a href="https://colab.research.google.com/github/Sofia888u/S3/blob/main/tp2_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from io import StringIO

from sklearn import neighbors, metrics, svm, datasets
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_graphviz


**Exercice 1 : KNN**

In [2]:
def exercice1_knn():
    # Chargement de MNIST
    mnist = fetch_openml('mnist_784', version=1, as_frame=False)
    print(mnist.data.shape)
    print(mnist.target.shape)

    # Echantillonnage de 5000 images
    sample = np.random.randint(70000, size=5000)
    data = mnist.data[sample]
    target = mnist.target[sample]

    # Séparation apprentissage / test (80% / 20%)
    xtrain, xtest, ytrain, ytest = train_test_split(data, target, train_size=0.8)

    # Modèle 3-NN
    knn = neighbors.KNeighborsClassifier(n_neighbors=3)
    knn.fit(xtrain, ytrain)
    error = 1 - knn.score(xtest, ytest)
    print('Erreur 3-NN : {:.3f}'.format(error))

    # Optimisation : k de 2 à 15
    errors = []
    for k in range(2, 16):
        knn = neighbors.KNeighborsClassifier(n_neighbors=k)
        knn.fit(xtrain, ytrain)
        errors.append(100 * (1 - knn.score(xtest, ytest)))
    plt.figure()
    plt.plot(range(2, 16), errors, 'o-')
    plt.xlabel('k')
    plt.ylabel('Erreur test (%)')
    plt.title('Erreur en fonction de k')
    plt.show()

    # Meilleur classifieur
    best_k = range(2, 16)[int(np.argmin(errors))]
    print('k optimal :', best_k)
    knn = neighbors.KNeighborsClassifier(n_neighbors=best_k)
    knn.fit(xtrain, ytrain)

    # Prédictions sur quelques images
    predicted = knn.predict(xtest)
    images = xtest.reshape((-1, 28, 28))
    select = np.random.randint(images.shape[0], size=12)

    fig, ax = plt.subplots(3, 4)
    for index, value in enumerate(select):
        plt.subplot(3, 4, index + 1)
        plt.axis('off')
        plt.imshow(images[value], cmap=plt.cm.gray_r, interpolation='nearest')
        plt.title('Predicted: {}'.format(predicted[value]))
    plt.show()



 **Exercice 2 : Arbre de décision**

In [3]:
def exercice2_arbre(csv_path='recrutement.csv'):
    data = pd.read_csv(csv_path)
    print(data.head(10))

    # Question : remplacer les chaînes par des valeurs numériques
    d_yn = {'Y': 1, 'N': 0}
    d_edu = {'BS': 0, 'MS': 1, 'PhD': 2}
    data['Employed?'] = data['Employed?'].map(d_yn)
    data['Top-tier school'] = data['Top-tier school'].map(d_yn)
    data['Interned'] = data['Interned'].map(d_yn)
    data['Hired'] = data['Hired'].map(d_yn)
    data['Level of Education'] = data['Level of Education'].map(d_edu)
    print(data.head(10))

    # Création du modèle
    X = data.iloc[:, :-1]
    y = data.iloc[:, -1]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

    model = DecisionTreeClassifier()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print('précision', metrics.accuracy_score(y_test, y_pred))

    # Visualisation de l'arbre (pydotplus + Graphviz requis)
    try:
        import pydotplus
        dot_data = StringIO()
        export_graphviz(model, out_file=dot_data, feature_names=X.columns)
        graph = pydotplus.graph_from_dot_data(dot_data.getvalue())
        graph.write_png('arbre.png')
        print("Arbre enregistré dans arbre.png")
    except Exception as e:
        # Alternative sans Graphviz
        from sklearn.tree import plot_tree
        plt.figure(figsize=(12, 7))
        plot_tree(model, feature_names=list(X.columns), filled=True)
        plt.savefig('arbre.png')
        print("pydotplus/Graphviz indisponible ({}), arbre.png créé avec plot_tree".format(e))


In [5]:
csv = """Years Experience,Employed?,Previous employers,Level of Education,Top-tier school,Interned,Hired
10,Y,4,BS,N,N,Y
0,N,0,BS,Y,Y,Y
7,N,6,BS,N,N,N
2,Y,1,MS,Y,N,Y
20,N,2,PhD,Y,N,N
0,N,0,PhD,Y,Y,Y
5,Y,2,MS,N,Y,Y
3,N,1,BS,N,Y,Y
15,Y,5,BS,N,N,Y
0,N,0,BS,N,N,N
"""
with open('recrutement.csv', 'w') as f:
    f.write(csv)

 **Exercice 3 : SVM**

In [6]:
def exercice3_svm():
    data = datasets.load_breast_cancer()
    print(data.keys())
    print(data.target_names)
    print(data.DESCR)  # Question : analyser la description

    X = data.data
    Y = data.target

    # Partition (avant standardisation pour éviter la fuite de données)
    X_train, X_test, y_train, y_test = train_test_split(
        X, Y, test_size=0.2, random_state=42)

    # Standardisation
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    def evaluer(nom, model):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        print('\n=== {} ==='.format(nom))
        print('Précision du modèle', metrics.accuracy_score(y_test, y_pred))
        print(metrics.confusion_matrix(y_test, y_pred))
        return y_pred

    # Noyau linéaire
    y_pred = evaluer('SVM linéaire', svm.SVC(kernel='linear'))
    comp = pd.DataFrame({'prediction': y_pred, 'reel': y_test})
    print(comp.head(10))

    # Noyau polynomial
    evaluer('SVM poly (degree=1, gamma=100)',
            svm.SVC(kernel='poly', degree=1, gamma=100))

    # Noyau sigmoid
    evaluer('SVM sigmoid (C=0.2)',
            svm.SVC(kernel='sigmoid', degree=200, C=0.2))


if __name__ == '__main__':
    # exercice1_knn()      # télécharge MNIST (internet requis)
    exercice2_arbre()
    exercice3_svm()


   Years Experience Employed?  Previous employers Level of Education  \
0                10         Y                   4                 BS   
1                 0         N                   0                 BS   
2                 7         N                   6                 BS   
3                 2         Y                   1                 MS   
4                20         N                   2                PhD   
5                 0         N                   0                PhD   
6                 5         Y                   2                 MS   
7                 3         N                   1                 BS   
8                15         Y                   5                 BS   
9                 0         N                   0                 BS   

  Top-tier school Interned Hired  
0               N        N     Y  
1               Y        Y     Y  
2               N        N     N  
3               Y        N     Y  
4               Y        N     N